# Technical Analysis Trading Agent with MCP

In this notebook I built:

1. `ta_server.py` – A custom MCP server exposing Polygon's TA tools (SMA, EMA, RSI, MACD, price bars, news)
2. `search_server.py` – A free DuckDuckGo search MCP server (no API key needed)
3. A Researcher agent – combines TA + search + fetch + memory for deep analysis
4. A Trader agent – uses the researcher as a sub-tool plus accounts for execution

## Setup & Imports

In [ ]:
import os
import sys
import asyncio
from pathlib import Path
from dotenv import load_dotenv
from agents import Agent, Runner, trace, Tool
from agents.mcp import MCPServerStdio
from IPython.display import Markdown, display
from datetime import datetime

# Resolve paths
# 6_mcp/ is the shared root with accounts, market, database, etc.
MCP_DIR = Path(os.path.abspath("")).resolve()
# Walk up until we find 6_mcp by looking for market.py
while MCP_DIR.name != "6_mcp" and MCP_DIR.parent != MCP_DIR:
    MCP_DIR = MCP_DIR.parent
if MCP_DIR.name != "6_mcp":
    # Fallback: assume we're running from the workspace root
    MCP_DIR = Path(os.path.abspath("6_mcp")).resolve()

# Our custom servers live here
MY_DIR = MCP_DIR / "community_contributions" / "EmmanuelSamuel"

# Add 6_mcp/ to sys.path so shared modules (accounts, market, etc.) are importable
if str(MCP_DIR) not in sys.path:
    sys.path.insert(0, str(MCP_DIR))

# Change CWD to 6_mcp/ so shared `uv run accounts_server.py` etc. work
os.chdir(MCP_DIR)
print(f"Working directory: {os.getcwd()}")
print(f"Custom servers in: {MY_DIR}")

# Windows Jupyter fix for MCPServerStdio
if sys.platform == "win32":
    from community_contributions.windows_no_wsl.winpatch import winpatch_mcpserver_stdio
    winpatch_mcpserver_stdio()

# Suppress anyio cancel-scope cleanup noise in Jupyter
_original_handler = asyncio.get_event_loop().get_exception_handler()
def _suppress_cancel_scope_error(loop, context):
    exc = context.get("exception")
    if exc and "cancel scope" in str(exc):
        return  # silently ignore
    if _original_handler:
        _original_handler(loop, context)
    else:
        loop.default_exception_handler(context)
asyncio.get_event_loop().set_exception_handler(_suppress_cancel_scope_error)

load_dotenv(override=True)
print("Setup complete")

## Explore the Technical Analysis MCP Server

In [ ]:
ta_params = {"command": "uv", "args": ["run", str(MY_DIR / "ta_server.py")]}

async with MCPServerStdio(params=ta_params, client_session_timeout_seconds=30) as server:
    ta_tools = await server.list_tools()

print(f"Technical Analysis server has {len(ta_tools)} tools:")
for t in ta_tools:
    print(f"  - {t.name}: {t.description[:80]}")

### Quick test – ask an agent to pull RSI for AAPL

In [ ]:
async with MCPServerStdio(params=ta_params, client_session_timeout_seconds=30) as server:
    agent = Agent(
        name="TA Tester",
        instructions="You are a technical analysis assistant. Use your tools to answer questions about stock indicators.",
        model="gpt-4.1-mini",
        mcp_servers=[server],
    )
    with trace("ta-test"):
        result = await Runner.run(agent, "What is the current RSI and MACD for AAPL? Is it overbought or oversold?")
    display(Markdown(result.final_output))

## Explore the DuckDuckGo Search MCP Server

In [ ]:
search_params = {"command": "uv", "args": ["run", str(MY_DIR / "search_server.py")]}

async with MCPServerStdio(params=search_params, client_session_timeout_seconds=30) as server:
    search_tools = await server.list_tools()

print(f"Search server has {len(search_tools)} tools:")
for t in search_tools:
    print(f"  - {t.name}: {t.description[:80]}")

In [ ]:
async with MCPServerStdio(params=search_params, client_session_timeout_seconds=30) as server:
    agent = Agent(
        name="Search Tester",
        instructions="You search the web to answer questions. Summarize your findings concisely.",
        model="gpt-4.1-mini",
        mcp_servers=[server],
    )
    with trace("search-test"):
        result = await Runner.run(agent, f"What are the latest headlines about NVDA stock? Today is {datetime.now().strftime('%Y-%m-%d')}")
    display(Markdown(result.final_output))

## Define All MCP Server

In [ ]:
# Technical Analysis MCP (our local Polygon TA wrapper)
ta_params = {"command": "uv", "args": ["run", str(MY_DIR / "ta_server.py")]}

# DuckDuckGo Search MCP (our local free search server)
search_params = {"command": "uv", "args": ["run", str(MY_DIR / "search_server.py")]}

# Fetch MCP (read full web pages via headless browser)
fetch_params = {"command": "uvx", "args": ["mcp-server-fetch"]}

# Memory MCP (persistent knowledge graph)
memory_params = {
    "command": "npx",
    "args": ["-y", "mcp-memory-libsql"],
    "env": {"LIBSQL_URL": f"file:{MY_DIR}/memory/ta_agent.db"},
}

# Accounts MCP (shared, from 6_mcp/)
accounts_params = {"command": "uv", "args": ["run", "accounts_server.py"]}

# Group them by role
researcher_params_list = [ta_params, search_params, fetch_params, memory_params]
trader_params_list = [accounts_params, ta_params]

print(f"Researcher will have {len(researcher_params_list)} MCP servers")
print(f"Trader will have {len(trader_params_list)} MCP servers")

## Build the Researcher Agent

In [ ]:
researcher_instructions = f"""You are a financial research analyst specializing in technical analysis.
You have access to tools for:
1. Technical indicators (SMA, EMA, RSI, MACD) and historical price data from Polygon.io
2. Web search via DuckDuckGo for the latest financial news
3. Fetching full web pages for detailed reading
4. A persistent knowledge graph to store and recall your findings

When asked to research a stock, you should:
- Pull the RSI (14-day) to check overbought/oversold conditions
- Pull the MACD to identify momentum and trend direction
- Pull SMA(20) and SMA(50) to check for golden/death crosses
- Search for recent news that might affect the stock
- Store key findings in your knowledge graph for future reference
- Synthesize everything into a clear buy/hold/sell recommendation with reasoning

Always cite the specific indicator values in your analysis.
The current datetime is {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
"""


async def create_researcher(mcp_servers):
    return Agent(
        name="TA Researcher",
        instructions=researcher_instructions,
        model="gpt-4.1-mini",
        mcp_servers=mcp_servers,
    )


async def create_researcher_tool(mcp_servers):
    researcher = await create_researcher(mcp_servers)
    return researcher.as_tool(
        tool_name="TechnicalResearcher",
        tool_description=(
            "This tool performs in-depth technical analysis on a stock, including RSI, MACD, "
            "moving averages, price history, and latest news. Describe what stock or analysis you need."
        ),
    )


print("Researcher agent defined")

### Test the Researcher standalone

In [ ]:
from contextlib import AsyncExitStack

async with AsyncExitStack() as stack:
    researcher_mcp_servers = [
        await stack.enter_async_context(MCPServerStdio(params=p, client_session_timeout_seconds=60))
        for p in researcher_params_list
    ]
    researcher = await create_researcher(researcher_mcp_servers)
    with trace("researcher-test"):
        result = await Runner.run(
            researcher,
            "Perform a full technical analysis on MSFT. Check RSI, MACD, moving averages and recent news. Give a buy/hold/sell recommendation.",
            max_turns=15,
        )

display(Markdown(result.final_output))

## Build the Trader Agent

In [ ]:
from accounts_client import read_accounts_resource, read_strategy_resource
from accounts import Account

# Reset the account with a fresh strategy
agent_name = "TATrader"
strategy = (
    "You are a swing trader that relies heavily on technical analysis. "
    "You buy when RSI is below 35 and MACD shows bullish crossover. "
    "You sell when RSI is above 70 or MACD shows bearish crossover. "
    "You use SMA(20) and SMA(50) crossovers to confirm trend direction. "
    "You diversify across 3-5 stocks and never put more than 30% in a single position."
)

Account.get(agent_name).reset(strategy)

display(Markdown(await read_accounts_resource(agent_name)))
display(Markdown(await read_strategy_resource(agent_name)))

In [ ]:
account_details = await read_accounts_resource(agent_name)
current_strategy = await read_strategy_resource(agent_name)

STOCK = "NVDA"  # change this to any ticker you want to deep-dive

trader_instructions = f"""You are {agent_name}, a senior technical-analysis-driven stock trader.
Your account is under the name {agent_name}.

Your investment strategy:
{current_strategy}

Your current holdings and balance:
{account_details}

You have direct access to tools for:
- Technical indicators: get_rsi, get_macd, get_sma, get_ema, get_price_history, get_previous_close
- Web & news search (DuckDuckGo): search_web, search_news
- Polygon news: get_ticker_news
- Trading: buy_shares, sell_shares, get_balance, get_holdings

You are analyzing **{STOCK}** only. Perform the deepest possible analysis using EVERY tool at your disposal.

### Required Analysis Steps

**1. Price Action & Trend**
- Get 30-day price history and describe the overall trend (uptrend, downtrend, consolidation)
- Note key support/resistance levels from recent highs and lows
- Get the previous close and compare to the 30-day range

**2. Moving Averages (Trend Confirmation)**
- Get SMA(20) and SMA(50) – identify golden cross or death cross
- Get EMA(12) and EMA(26) – confirm short-term momentum direction
- Note whether price is trading above or below each moving average

**3. RSI (Momentum & Extremes)**
- Get RSI(14) and interpret: overbought (>70), oversold (<30), or neutral
- Look at the RSI trend over the last several data points – is it rising or falling?

**4. MACD (Momentum & Crossovers)**
- Get MACD with default settings (12, 26, 9)
- Identify if MACD line is above or below the signal line (bullish vs bearish)
- Check if histogram is expanding or contracting (strengthening or weakening momentum)
- Note any recent crossover

**5. News & Sentiment**
- Use search_news to find the latest news about {STOCK}
- Use get_ticker_news to get Polygon's curated news
- Summarize the overall sentiment: bullish, bearish, or mixed
- Highlight any upcoming catalysts (earnings, product launches, regulatory events)

**6. Trading Decision**
- Synthesize all indicators into a clear BUY / HOLD / SELL decision
- If BUY: specify how many shares and why (cite the indicators)
- If SELL: specify how many shares and why
- If HOLD: explain what would change your mind
- Execute the trade using buy_shares or sell_shares

### Final Report Format (Markdown)

Your final response MUST be a detailed markdown report with these sections:

## {STOCK} – Deep Technical Analysis

### 1. Price Action & Trend
(30-day price summary, support/resistance, previous close, trend direction)

### 2. Moving Averages
| Indicator | Value | Interpretation |
|-----------|-------|----------------|
| SMA(20)   | ...   | ...            |
| SMA(50)   | ...   | ...            |
| EMA(12)   | ...   | ...            |


(Golden/death cross status, price vs MAs)

### 3. RSI Analysis
(RSI value, trend, overbought/oversold interpretation)

### 4. MACD Analysis
(MACD value, signal, histogram, crossover status, momentum assessment)

### 5. News & Sentiment
(Key headlines, overall sentiment, upcoming catalysts)

### 6. Trading Decision
(Final verdict with full reasoning, trade executed, position sizing rationale)

### 7. Risk Assessment
(What could go wrong, key levels to watch, stop-loss suggestion)

The current datetime is {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
"""

trade_prompt = f"""Perform a comprehensive deep-dive analysis on {STOCK} using every technical tool available.

Go through ALL of these steps:
1. Get 30-day price history – describe the trend, support, resistance
2. Get previous close – where did it finish?
3. Get SMA(20), SMA(50), EMA(12) – moving average analysis
4. Get RSI(14) – momentum and overbought/oversold
5. Get MACD – crossover and momentum strength
6. Search for latest news with search_news AND get_ticker_news
7. Make your BUY / HOLD / SELL decision and execute the trade

Present your findings as a detailed markdown report with tables for indicator values."""

print(f"Trader instructions ready for {STOCK} deep-dive")

## Run the Full Trading Agent

In [ ]:
from contextlib import AsyncExitStack

all_params = [
    {"command": "uv", "args": ["run", "accounts_server.py"]},                   # buy/sell/balance (shared)
    {"command": "uv", "args": ["run", str(MY_DIR / "ta_server.py")]},            # RSI, MACD, SMA, EMA, prices (local)
    {"command": "uv", "args": ["run", str(MY_DIR / "search_server.py")]},        # DuckDuckGo search (local)
]

async with AsyncExitStack() as stack:
    all_servers = [
        await stack.enter_async_context(MCPServerStdio(params=p, client_session_timeout_seconds=3000))
        for p in all_params
    ]

    trader = Agent(
        name=agent_name,
        instructions=trader_instructions,
        model="gpt-4.1-mini",
        mcp_servers=all_servers,
    )

    print(f"Trader '{agent_name}' ready with {len(all_servers)} MCP servers (flat, no sub-agent)")

    with trace(f"{agent_name}-trading"):
        result = await Runner.run(trader, trade_prompt, max_turns=20)

display(Markdown(result.final_output))

## Research a Single Stock

Use the researcher standalone to do a thorough TA report on any stock you choose.

In [ ]:
from contextlib import AsyncExitStack

STOCK = "TSLA"  # change this to any ticker

deep_dive_prompt = f"""Perform a comprehensive technical analysis on {STOCK}:
1. Get 30-day price history and describe the trend
2. Check RSI(14) – is it overbought or oversold?
3. Check MACD – is there a bullish or bearish crossover?
4. Compare SMA(20) vs SMA(50) – golden cross or death cross?
5. Search for the latest news about {STOCK}
6. Give a final BUY / HOLD / SELL recommendation with confidence level
"""

async with AsyncExitStack() as stack:
    servers = [
        await stack.enter_async_context(MCPServerStdio(params=p, client_session_timeout_seconds=60))
        for p in researcher_params_list
    ]
    researcher = await create_researcher(servers)
    with trace(f"deep-dive-{STOCK}"):
        result = await Runner.run(researcher, deep_dive_prompt, max_turns=15)

display(Markdown(result.final_output))